# 03 — Publish Gold data products

**Before running:** run Notebook 02 successfully. This notebook aggregates governed Silver trip records into daily and pickup-zone metrics for use through the Lakehouse SQL analytics endpoint and Power BI.


In [ ]:
from pyspark.sql import functions as F

silver = spark.table("silver_yellow_taxi_trips")

# Daily fact-style product. total_amount is the source-reported passenger charge,
# which excludes cash tips according to the TLC data dictionary.
gold_daily = (silver
    .groupBy("trip_date")
    .agg(
        F.count(F.lit(1)).alias("trip_count"),
        F.round(F.sum("total_amount"), 2).alias("recorded_total_amount"),
        F.round(F.avg("fare_amount"), 2).alias("average_fare_amount"),
        F.round(F.avg("tip_amount"), 2).alias("average_recorded_tip_amount"),
        F.round(F.avg("trip_distance"), 2).alias("average_trip_distance_miles"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("average_trip_duration_minutes"),
    )
    .withColumn("published_at_utc", F.current_timestamp()))

(gold_daily.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_daily_metrics"))

# TLC lookup table columns are LocationID, Borough, Zone, service_zone.
zones_raw = spark.table("bronze_taxi_zone_lookup")
zone_cols = {c.lower(): c for c in zones_raw.columns}
required_zone_cols = {"locationid", "borough", "zone"}
if not required_zone_cols.issubset(set(zone_cols)):
    raise ValueError(f"Taxi zone lookup schema not recognized. Found columns: {zones_raw.columns}")
zones = zones_raw.select(
    F.col(zone_cols["locationid"]).cast("int").alias("pickup_location_id"),
    F.col(zone_cols["borough"]).cast("string").alias("pickup_borough"),
    F.col(zone_cols["zone"]).cast("string").alias("pickup_zone"),
)

gold_zone = (silver
    .groupBy("pickup_location_id")
    .agg(
        F.count(F.lit(1)).alias("trip_count"),
        F.round(F.sum("total_amount"), 2).alias("recorded_total_amount"),
        F.round(F.avg("fare_amount"), 2).alias("average_fare_amount"),
        F.round(F.avg("trip_distance"), 2).alias("average_trip_distance_miles"),
    )
    .join(zones, on="pickup_location_id", how="left")
    .withColumn("pickup_borough", F.coalesce(F.col("pickup_borough"), F.lit("Unknown")))
    .withColumn("pickup_zone", F.coalesce(F.col("pickup_zone"), F.lit("Unknown")))
    .withColumn("published_at_utc", F.current_timestamp())
    .select("pickup_location_id", "pickup_borough", "pickup_zone", "trip_count", "recorded_total_amount", "average_fare_amount", "average_trip_distance_miles", "published_at_utc"))

(gold_zone.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_pickup_zone_metrics"))

print("Gold daily rows:", spark.table("gold_daily_metrics").count())
print("Gold pickup-zone rows:", spark.table("gold_pickup_zone_metrics").count())
display(spark.table("gold_daily_metrics").orderBy("trip_date"))
display(spark.table("gold_pickup_zone_metrics").orderBy(F.desc("trip_count")).limit(20))
